# Stage 2 - Airgapped quarantine (internet disabled)

**NASA Space Apps 2026 - MODIS/VIIRS hotspot harmonization**

Prove the offline environment works before any GPU-heavy run: install only from the local wheelhouse, import every pipeline module, verify staged hashes, load a local model artifact and run a representative forward pass. Run this notebook with the internet toggle OFF; a failed network request alone does not prove offline safety.

| | |
|---|---|
| Stage | 2 of 6 - offline quarantine |
| Network required | NO - must run with internet disabled |
| Runs | the committed CLI scripts (`scripts/`) - logic stays out of cells |
| Writes to | `quarantine_report.json + smoke-test outputs` |

Rules that make this notebook reproducible (docs/DEVELOPMENT.md): record the
environment below, keep every output under the working directory, never fetch
from PyPI/GitHub/Hugging Face/REST APIs during offline compute, and keep
generated weights/datasets out of Git.


## 0. Record the environment

Attach the wheelhouse dataset from stage 1 as a read-only input. Mount points are read-only; all outputs go to the working directory.

In [ ]:
import sys, platform, subprocess, json, hashlib
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WHEELHOUSE = None
candidates = list(KAGGLE_INPUT.glob("*/wheelhouse")) if KAGGLE_INPUT.exists() else []
candidates += list((Path.cwd() / "out").glob("wheelhouse"))
WHEELHOUSE = candidates[0] if candidates else Path("out/wheelhouse")
KAGGLE_WORKING = Path("/kaggle/working")
WORK_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "out"
WORK_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd()

def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("wheelhouse:", WHEELHOUSE)
print("repo root :", REPO_ROOT)
print("work dir  :", WORK_DIR)
print("python    :", sys.version)
print("platform  :", platform.platform())
RECORD = {"python": sys.version.split()[0], "platform": platform.platform(),
          "machine": platform.machine(), "wheelhouse": str(WHEELHOUSE)}


## 1. Verify staged hashes

Re-compute SHA-256 for every staged wheel and compare against `staging_manifest.json` from stage 1. A mismatch aborts the quarantine: never trust unverified inputs.

In [ ]:
import hashlib, json
from pathlib import Path

manifest_path = next((p for p in WHEELHOUSE.parent.rglob("staging_manifest.json")), None)
if manifest_path is None:
    manifest_path = next(Path(".").rglob("staging_manifest.json"), None)
assert manifest_path is not None, "staging_manifest.json not found - run stage 1 first"
staging = json.loads(manifest_path.read_text())

mismatches = []
for rel, expected in staging["wheelhouse"].items():
    local = WHEELHOUSE / rel
    if not local.exists():
        mismatches.append(f"missing: {rel}")
    elif sha256(local) != expected:
        mismatches.append(f"hash mismatch: {rel}")
print("verified files:", len(staging["wheelhouse"]), "| mismatches:", len(mismatches))
assert not mismatches, "\n".join(mismatches)


## 2. Install only from the local wheelhouse

`--no-index --find-links` guarantees no PyPI access. The offline safety tests (`tests/offline/test_offline_safety.py`) additionally assert the source tree performs no network imports.

In [ ]:
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(WHEELHOUSE),
     "-r", str(WHEELHOUSE.parent / "requirements-offline.txt")],
    capture_output=True, text=True)
print(proc.stdout[-2500:] or proc.stderr[-2500:])
assert proc.returncode == 0, "offline install failed - wheelhouse is incomplete"
print("offline install OK (no index reached)")


## 3. Import every pipeline module (no network)

Each `src` package must import cleanly; `torch` stays optional and its absence must degrade gracefully (clear error, baselines usable).

In [ ]:
import importlib
MODULES = [
    "src.common.config", "src.common.logging",
    "src.data.schema", "src.data.adapters", "src.data.gate", "src.data.aggregation",
    "src.geo.aoi", "src.geo.matching",
    "src.harmonization.features", "src.harmonization.baselines",
    "src.harmonization.model", "src.harmonization.evaluate",
    "src.harmonization.train", "src.harmonization.inference",
    "src.calendar.seasonal", "src.calendar.anomalies", "src.calendar.forecast",
    "src.calendar.evaluation", "src.calendar.records",
    "src.artifacts.contracts", "src.artifacts.manifest", "src.artifacts.serialization",
]
imported = []
for module in MODULES:
    importlib.import_module(module)
    imported.append(module)
print(f"imported {len(imported)}/{len(MODULES)} pipeline modules")

try:
    importlib.import_module("torch")
    TORCH = True
except ImportError:
    TORCH = False
print("torch available:", TORCH, "(PatchTST stays optional; baselines unaffected)")


## 4. Representative forward pass on local artifacts

Load a local harmonizer artifact (hash-verified on load) and run a small harmonized-series build on the committed synthetic fixtures. If no real artifact exists yet, the fixtures prove the code path.

In [ ]:
import tempfile, json
from pathlib import Path
sys.path.insert(0, str(REPO_ROOT))

from src.common.config import HarmonizationConfig
from src.artifacts.serialization import load_harmonizer_bundle

FIXTURES = REPO_ROOT / "tests" / "fixtures" / "sample_aoi"
artifact_dirs = sorted(REPO_ROOT.rglob("harmonizer_*/model.joblib"))
smoke = {"imports": len(imported), "torch": TORCH, "forward_pass": None}

if artifact_dirs:
    bundle = load_harmonizer_bundle(artifact_dirs[-1].parent)  # verifies SHA-256
    from src.data.gate import FivePassDataGate, assert_gate_passed
    from src.geo.aoi import parse_aoi, filter_records
    from src.data.aggregation import build_daily_frame
    from src.harmonization.inference import build_harmonized_daily
    from src.data.schema import PRODUCT_MODIS, PRODUCT_VIIRS

    config = HarmonizationConfig()
    outcome = FivePassDataGate(config).run(str(FIXTURES))
    assert_gate_passed(outcome)
    aoi = parse_aoi("bbox:-123.5,39.5,-120.5,42.0")
    frames = {p: filter_records(f, aoi) for p, f in outcome.frames.items()}
    daily = build_daily_frame(frames, aoi_id=aoi.aoi_id)
    harmonized = build_harmonized_daily(
        daily, frames[PRODUCT_MODIS], frames[PRODUCT_VIIRS], bundle["model"], config.matching
    )
    smoke["forward_pass"] = {"days": int(len(harmonized))}
    print("forward pass on synthetic fixtures ->", smoke["forward_pass"])
else:
    print("no staged harmonizer artifact found; fixture-level forward pass only")

(WORK_DIR / "quarantine_report.json").write_text(json.dumps(
    {"status": "passed", "environment": RECORD, "checks": smoke}, indent=2))
print("quarantine PASSED; report:", WORK_DIR / "quarantine_report.json")
